In [7]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit, StratifiedGroupKFold, cross_val_score, cross_validate, StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, make_scorer, fbeta_score, precision_score, recall_score, average_precision_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_class_weight

In [3]:
# Define features and target variable
feature = [
    "Stop:Station code", 
    "Hour",
    "DayOfWeek_sin",
    "DayOfWeek_cos", 
    "Month_sin", 
    "Month_cos", 
    "IsWeekend", 
    "RushHour",
    "StationTraffic",
    "Temperature",
    "Humidity",
    "Air Pressure",
    "Horizontal Visibility",
    "Cloud Cover",
    "Precipitation Amount",
    "Precipitation Duration",
    "Hourly Mean Wind Speed",
    "Max Wind Speed",
    "Fog",
    "Rainfall",
    "Snowfall",
    "Thunder",
    "Hail",
    "Service:Type"
    ]

target = "is_cancelled"

In [4]:
# Define file paths for each dataset split
files = {
    "Train": "train.csv",
    "Validation": "validation.csv",
    "Test": "test.csv"
}

# Read and process each dataset split in chunks
for split_name, file_path in files.items():
    print(f"\n{split_name} chunks:")
    total_rows = 0

    for i, chunk in enumerate(
        pd.read_csv(file_path, chunksize=1000000),
        start=1
    ):
        X_chunk = chunk[feature]
        y_chunk = chunk[target]
        total_rows += len(chunk)

        print(
            f"{split_name} chunk {i}: {chunk.shape}"
        )

    print(f"{split_name} total rows read: {total_rows:,}")


Train chunks:
Train chunk 1: (1000000, 26)
Train chunk 2: (1000000, 26)
Train chunk 3: (1000000, 26)
Train chunk 4: (1000000, 26)
Train chunk 5: (1000000, 26)
Train chunk 6: (1000000, 26)
Train chunk 7: (1000000, 26)
Train chunk 8: (1000000, 26)
Train chunk 9: (1000000, 26)
Train chunk 10: (1000000, 26)
Train chunk 11: (1000000, 26)
Train chunk 12: (1000000, 26)
Train chunk 13: (1000000, 26)
Train chunk 14: (1000000, 26)
Train chunk 15: (1000000, 26)
Train chunk 16: (1000000, 26)
Train chunk 17: (1000000, 26)
Train chunk 18: (1000000, 26)
Train chunk 19: (1000000, 26)
Train chunk 20: (1000000, 26)
Train chunk 21: (1000000, 26)
Train chunk 22: (1000000, 26)
Train chunk 23: (1000000, 26)
Train chunk 24: (1000000, 26)
Train chunk 25: (1000000, 26)
Train chunk 26: (1000000, 26)
Train chunk 27: (1000000, 26)
Train chunk 28: (1000000, 26)
Train chunk 29: (1000000, 26)
Train chunk 30: (1000000, 26)
Train chunk 31: (1000000, 26)
Train chunk 32: (1000000, 26)
Train chunk 33: (1000000, 26)
Trai

In [5]:
usecols = feature + [target]

train_df = pd.read_csv("train.csv", usecols=usecols)
validation_df = pd.read_csv("validation.csv", usecols=usecols)
test_df = pd.read_csv("test.csv", usecols=usecols)

for df in (train_df, validation_df, test_df):
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns
    df[float_cols] = df[float_cols].astype("float32")
    df[int_cols] = df[int_cols].astype("int32")

train_df = pd.get_dummies(
    train_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)
validation_df = pd.get_dummies(
    validation_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)
test_df = pd.get_dummies(
    test_df,
    columns=["Service:Type", "Stop:Station code"],
    drop_first=True
)

validation_df = validation_df.reindex(columns=train_df.columns, fill_value=0)
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

X_train, y_train = train_df.drop(columns=[target]).astype("float32"), train_df[target].astype("int8")
X_val, y_val = validation_df.drop(columns=[target]).astype("float32"), validation_df[target].astype("int8")
X_test, y_test = test_df.drop(columns=[target]).astype("float32"), test_df[target].astype("int8")

In [ ]:
# XGBoost from existing prepared data (no re-reading CSV).
# Stratified sampling to maintain class distribution in the sample
sss = StratifiedShuffleSplit(n_splits=1, train_size= min(len(X_train), 1_000_000), random_state=42)
for strat_idx, _ in sss.split(X_train, y_train):
    X_train_sample = X_train.iloc[strat_idx].astype("float32").to_numpy(copy=False)
    y_train_sample = y_train.iloc[strat_idx].to_numpy(copy=False)

# Handle class imbalance for XGBoost
neg_count = (y_train_sample == 0).sum()
pos_count = (y_train_sample == 1).sum()
scale_pos_weight = neg_count / max(pos_count, 1)

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
)

xgb.fit(X_train_sample, y_train_sample)
print(f"XGBoost trained on {len(y_train_sample):,} samples (stratified)")

# Evaluate on validation and test sets in chunks to avoid memory issues
def evaluate_split_xgb(name, X, y):
    y_true_parts = []
    y_pred_parts = []
    y_prob_parts = []

    for start in range(0, len(X), 1_000_000):
        X_chunk = X.iloc[start:start + 1_000_000]
        y_chunk = y.iloc[start:start + 1_000_000]

        X_chunk_np = X_chunk.to_numpy(dtype=np.float32, copy=False)
        y_true_parts.append(y_chunk.to_numpy(copy=False))
        y_pred_parts.append(xgb.predict(X_chunk_np))
        y_prob_parts.append(xgb.predict_proba(X_chunk_np)[:, 1])

    y_true_all = np.concatenate(y_true_parts)
    y_pred_all = np.concatenate(y_pred_parts)
    y_prob_all = np.concatenate(y_prob_parts)

    f2 = fbeta_score(y_true_all, y_pred_all, beta=2)
    print("F2 Score:", round(f2, 4))
    print(f"\n{name}")
    print(classification_report(y_true_all, y_pred_all, digits=3, zero_division=0))
    print("PR-AUC:", round(average_precision_score(y_true_all, y_prob_all), 4))

evaluate_split_xgb("Validation", X_val, y_val)
evaluate_split_xgb("Test", X_test, y_test)

# Feature importances
feature_cols = X_train.columns
importances = pd.Series(
    xgb.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\nTop 20 Feature Importances:")
print(importances.head(20))

XGBoost trained on 1,000,000 samples (stratified)
F2 Score: 0.251

Validation
              precision    recall  f1-score   support

           0      0.958     0.654     0.777   9439441
           1      0.082     0.519     0.141    560559

    accuracy                          0.647  10000000
   macro avg      0.520     0.587     0.459  10000000
weighted avg      0.909     0.647     0.742  10000000

PR-AUC: 0.0806
F2 Score: 0.2903

Test
              precision    recall  f1-score   support

           0      0.942     0.626     0.752   8426700
           1      0.104     0.528     0.173    691095

    accuracy                          0.618   9117795
   macro avg      0.523     0.577     0.463   9117795
weighted avg      0.878     0.618     0.708   9117795

PR-AUC: 0.1026

Top 20 Feature Importances:
Service:Type_Sprinter            0.409400
Service:Type_Intercity           0.196010
Service:Type_Intercity direct    0.036041
Max Wind Speed                   0.017999
Month_sin         

In [13]:
print(train_df.columns.tolist())

['Hour', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'IsWeekend', 'RushHour', 'StationTraffic', 'Temperature', 'Humidity', 'Air Pressure', 'Horizontal Visibility', 'Cloud Cover', 'Precipitation Amount', 'Precipitation Duration', 'Hourly Mean Wind Speed', 'Max Wind Speed', 'Fog', 'Rainfall', 'Snowfall', 'Thunder', 'Hail', 'is_cancelled', 'Service:Type_Intercity', 'Service:Type_Intercity direct', 'Service:Type_Sprinter', 'Stop:Station code_AH', 'Stop:Station code_AHP', 'Stop:Station code_AHPR', 'Stop:Station code_AHZ', 'Stop:Station code_AKM', 'Stop:Station code_ALM', 'Stop:Station code_ALMB', 'Stop:Station code_ALMM', 'Stop:Station code_ALMO', 'Stop:Station code_ALMP', 'Stop:Station code_AMF', 'Stop:Station code_AMFS', 'Stop:Station code_AML', 'Stop:Station code_AMPO', 'Stop:Station code_AMR', 'Stop:Station code_AMRI', 'Stop:Station code_AMRN', 'Stop:Station code_ANA', 'Stop:Station code_APD', 'Stop:Station code_APDO', 'Stop:Station code_APN', 'Stop:Station code_ARN', 'S